In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Enable GPU T4 in Kaggle settings."

class AttentionGate(nn.Module):
    def __init__(self, gate_ch, skip_ch, hidden_ch):
        super().__init__()
        self.gate = nn.Conv2d(gate_ch, hidden_ch, 1)
        self.skip = nn.Conv2d(skip_ch, hidden_ch, 1)
        self.psi = nn.Conv2d(hidden_ch, 1, 1)

    def forward(self, gate, skip):
        gate = F.interpolate(
            gate, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        attention = torch.sigmoid(
            self.psi(F.relu(self.gate(gate) + self.skip(skip)))
        )
        return skip * attention


class AttentionDecoder(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.attention = AttentionGate(in_ch, skip_ch, out_ch)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        attended = self.attention(x, skip)
        x = F.interpolate(
            x, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, attended], dim=1))


class SolarAttentionUNetV6C(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = resnet34(weights=ResNet34_Weights.DEFAULT)

        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu
        )
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.dec4 = AttentionDecoder(512, 256, 256)
        self.dec3 = AttentionDecoder(256, 128, 128)
        self.dec2 = AttentionDecoder(128, 64, 64)
        self.dec1 = AttentionDecoder(64, 64, 32)

        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        size = x.shape[-2:]

        f0 = self.stem(x)
        f1 = self.layer1(self.pool(f0))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)

        y = self.dec4(f4, f3)
        y = self.dec3(y, f2)
        y = self.dec2(y, f1)
        y = self.dec1(y, f0)

        y = F.interpolate(
            y, size=size,
            mode="bilinear", align_corners=False
        )

        return self.head(y)


model_v6c = SolarAttentionUNetV6C().to(device)
model_v6c.eval()

with torch.inference_mode():
    x = torch.randn(1, 3, 512, 512, device=device)
    y = model_v6c(x)

print("V6-C SPATIAL ATTENTION U-NET")
print("-" * 50)
print("Device:", device)
print("Input:", tuple(x.shape))
print("Output:", tuple(y.shape))
print("Parameters:", f"{sum(p.numel() for p in model_v6c.parameters()):,}")

assert y.shape == (1, 1, 512, 512)
print("PASS: V6-C attention architecture initialized.")

In [ ]:
import json, time
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(2026)

root = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/train"
)
image_dir = root / "train_images"

with open(root / "MAGFiLO_1.0_Annotations_kaggle2026_train.json") as f:
    coco = json.load(f)

anns_by_id = {}
for ann in coco["annotations"]:
    anns_by_id.setdefault(ann["image_id"], []).append(ann)

# Match the deterministic single-expert selection from V6-B.
records = {}
for rec in coco["images"]:
    name = rec["file_name"]
    if not (image_dir / name).is_file():
        continue

    previous = records.get(name)
    if previous is None:
        records[name] = rec
    else:
        new_count = len(anns_by_id.get(rec["id"], []))
        old_count = len(anns_by_id.get(previous["id"], []))
        if new_count > old_count or (
            new_count == old_count
            and str(rec["id"]) < str(previous["id"])
        ):
            records[name] = rec

train_names, val_names = train_test_split(
    sorted(records), test_size=0.2, random_state=42
)

class FilamentCrops(Dataset):
    def __init__(self, names):
        self.names = list(names)

    def __len__(self):
        return len(self.names) * 4

    def __getitem__(self, idx):
        name = self.names[idx // 4]
        anns = anns_by_id.get(records[name]["id"], [])

        with Image.open(image_dir / name) as im:
            rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

        h, w = rgb.shape[:2]
        canvas = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(canvas)

        for ann in anns:
            for poly in ann.get("segmentation", []):
                pts = list(zip(poly[0::2], poly[1::2]))
                if len(pts) >= 3:
                    draw.polygon(pts, fill=1)

        mask = np.asarray(canvas, dtype=np.uint8)

        if anns and np.random.random() < 0.75:
            ann = anns[np.random.randint(len(anns))]
            x, y, bw, bh = ann["bbox"]
            left = int(np.clip(x + bw/2 - 256, 0, w - 512))
            top = int(np.clip(y + bh/2 - 256, 0, h - 512))
        else:
            left = np.random.randint(0, w - 511)
            top = np.random.randint(0, h - 511)

        rgb = rgb[top:top+512, left:left+512]
        mask = mask[top:top+512, left:left+512]

        image = torch.from_numpy(
            np.ascontiguousarray(rgb.transpose(2, 0, 1))
        ).float() / 255.0

        target = torch.from_numpy(
            np.ascontiguousarray(mask[None])
        ).float()

        return image, target

loader_v6c = DataLoader(
    FilamentCrops(train_names),
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

def loss_v6c(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    p = torch.sigmoid(logits)
    inter = (p * targets).sum((1, 2, 3))
    den = p.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    dice = 1 - ((2 * inter + 1) / (den + 1)).mean()
    return bce + dice

optimizer_v6c = torch.optim.AdamW([
    {"params": model_v6c.stem.parameters(), "lr": 1e-5},
    {"params": model_v6c.layer1.parameters(), "lr": 1e-5},
    {"params": model_v6c.layer2.parameters(), "lr": 1e-5},
    {"params": model_v6c.layer3.parameters(), "lr": 1e-5},
    {"params": model_v6c.layer4.parameters(), "lr": 1e-5},
    {"params": model_v6c.dec4.parameters(), "lr": 1e-4},
    {"params": model_v6c.dec3.parameters(), "lr": 1e-4},
    {"params": model_v6c.dec2.parameters(), "lr": 1e-4},
    {"params": model_v6c.dec1.parameters(), "lr": 1e-4},
    {"params": model_v6c.head.parameters(), "lr": 1e-4}
], weight_decay=1e-4)

scaler_v6c = torch.amp.GradScaler("cuda")
model_v6c.train()

print("V6-C TRAINING")
print("Train:", len(train_names), "| Validation:", len(val_names))

for epoch in range(1, 4):
    losses = []
    start = time.time()

    for images, masks in loader_v6c:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer_v6c.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model_v6c(images)
            loss = loss_v6c(logits, masks)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Invalid loss in epoch {epoch}")

        scaler_v6c.scale(loss).backward()
        scaler_v6c.step(optimizer_v6c)
        scaler_v6c.update()
        losses.append(float(loss.item()))

    torch.save(
        model_v6c.state_dict(),
        f"/kaggle/working/v6c_attention_epoch{epoch}.pth"
    )

    print(
        f"Epoch {epoch}/3 | "
        f"Loss: {np.mean(losses):.4f} | "
        f"Time: {(time.time()-start)/60:.2f} min"
    )

assert len(train_names) == 565 and len(val_names) == 142
print("PASS: V6-C training completed; all three checkpoints saved.")

In [ ]:
import numpy as np
import torch
import cv2
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

model_v6c.eval()

evaluation_names = [
    "20110421164914Mh.jpeg",
    "20220718185352Mh.jpeg",
    "20111226155454Bh.jpeg",
    "20141007224934Lh.jpeg",
    "20160726135354Bh.jpeg",
    "20130502063134Lh.jpeg",
    "20110823120334Ch.jpeg",
    "20140116180214Mh.jpeg",
    "20130510134654Bh.jpeg",
    "20150112081954Uh.jpeg"
]

assert set(evaluation_names).issubset(set(val_names))

def tile_positions(length, tile=512, stride=256):
    positions = list(range(0, length - tile + 1, stride))
    if positions[-1] != length - tile:
        positions.append(length - tile)
    return positions

def predict_v6c(name):
    with Image.open(image_dir / name) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

    h, w = rgb.shape[:2]

    probability_sum = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.uint8)

    locations = []
    crops = []

    def flush():
        if not crops:
            return

        batch = torch.from_numpy(
            np.ascontiguousarray(np.stack(crops))
        ).float().to(device) / 255.0

        with torch.inference_mode(), torch.autocast(
            "cuda", dtype=torch.float16
        ):
            probabilities = torch.sigmoid(model_v6c(batch))

        probabilities = probabilities[:, 0].float().cpu().numpy()

        for p, (left, top) in zip(probabilities, locations):
            probability_sum[top:top+512, left:left+512] += p
            count[top:top+512, left:left+512] += 1

        crops.clear()
        locations.clear()

    for top in tile_positions(h):
        for left in tile_positions(w):
            crops.append(
                rgb[top:top+512, left:left+512].transpose(2, 0, 1)
            )
            locations.append((left, top))

            if len(crops) == 4:
                flush()

    flush()

    prob = probability_sum / np.maximum(count, 1)

    binary = (prob >= 0.55).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)

    binary = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )
    binary = cv2.morphologyEx(
        binary, cv2.MORPH_CLOSE, kernel
    )

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    predictions = []

    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])

        if area < 500:
            continue

        component = labels == i

        if float(prob[component].mean()) < 0.80:
            continue

        predictions.append(component)

    return predictions

def ground_truth_v6c(name):
    anns = anns_by_id.get(records[name]["id"], [])
    gt = []

    with Image.open(image_dir / name) as im:
        w, h = im.size

    for ann in anns:
        canvas = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(canvas)

        for polygon in ann.get("segmentation", []):
            points = list(zip(polygon[0::2], polygon[1::2]))

            if len(points) >= 3:
                draw.polygon(points, fill=1)

        mask = np.asarray(canvas, dtype=bool)

        if mask.any():
            gt.append(mask)

    return gt

def evaluate_v6c(pred, gt):
    matrix = np.zeros((len(pred), len(gt)), dtype=np.float32)
    gt_areas = [int(m.sum()) for m in gt]

    for i, p in enumerate(pred):
        p_area = int(p.sum())

        for j, g in enumerate(gt):
            inter = np.logical_and(p, g).sum()
            union = p_area + gt_areas[j] - inter
            matrix[i, j] = inter / union if union else 0.0

    valid = matrix >= 0.5

    rows, cols = linear_sum_assignment(
        -(valid.astype(np.float64) * 2 + matrix)
    )

    matched = [
        float(matrix[r, c])
        for r, c in zip(rows, cols)
        if valid[r, c]
    ]

    tp = len(matched)

    return np.array([
        tp,
        len(pred) - tp,
        len(gt) - tp,
        sum(matched)
    ], dtype=float)

totals = np.zeros(4, dtype=float)

print("V6-C FULL-RESOLUTION EVALUATION")
print("-" * 65)

for idx, name in enumerate(evaluation_names, 1):
    pred = predict_v6c(name)
    gt = ground_truth_v6c(name)

    result = evaluate_v6c(pred, gt)
    totals += result

    print(
        f"{idx:02d}/10 | GT={len(gt):2d} | "
        f"Pred={len(pred):2d} | TP={int(result[0]):2d}"
    )

tp, fp, fn, matched_iou = totals
denominator = tp + 0.5 * fp + 0.5 * fn
pq = matched_iou / denominator if denominator else 0.0

print("\nAGGREGATED RESULTS")
print("-" * 45)
print("TP:", int(tp))
print("FP:", int(fp))
print("FN:", int(fn))
print("PQ:", round(float(pq), 4))
print("V6-B epoch-3 reference PQ: 0.3302")
print("\nPASS: V6-C evaluation completed.")

In [ ]:
import json
import shutil
from pathlib import Path

source = Path("/kaggle/working/v6c_attention_epoch3.pth")
folder = Path("/kaggle/working/v6c_final_research")
folder.mkdir(parents=True, exist_ok=True)

assert source.is_file()

checkpoint = folder / "v6c_attention_best.pth"
shutil.copy2(source, checkpoint)

results = {
    "architecture": "V6-C ResNet34 Attention U-Net",
    "inspiration": "Contrails spatial attention",
    "selected_epoch": 3,
    "validation_images": 10,
    "ground_truth_instances": 92,
    "postprocessing": {
        "threshold": 0.55,
        "minimum_area": 500,
        "minimum_confidence": 0.80
    },
    "results": {
        "TP": 49,
        "FP": 68,
        "FN": 43,
        "PQ": 0.2998
    },
    "comparison": {
        "V6A_merged_PQ": 0.1998,
        "V6B_PQ": 0.3302,
        "V6C_PQ": 0.2998
    },
    "limitations": (
        "Ten selected validation images; "
        "not official Kaggle PQ."
    )
}

with open(folder / "experiment.json", "w") as f:
    json.dump(results, f, indent=2)

print("V6-C CHECKPOINT SAVED")
print("Checkpoint:", checkpoint)
print("Size:", round(checkpoint.stat().st_size / 1024**2, 2), "MB")
print("PQ:", results["results"]["PQ"])
print("PASS: V6-C research experiment preserved.")